In [7]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# 1. API 키 환경변수 세팅
load_dotenv()

# 2. LLM 세팅 (모델은 변경 가능)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# 3. 임베딩 모델 세팅 (텍스트를 벡터로 바꿔줄 모델)
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# 연결 테스트
print("LLM 테스트 중...")
response = llm.invoke("안녕? 넌 어떤 모델이니?")
print("LLM 답변: ", response.content)

print("\n임베딩 테스트 중...")
vector = embeddings.embed_query("경북대학교 컴퓨터학부 졸업요건")
print(f"임베딩 완료! 텍스트가 {len(vector)} 차원의 숫자 벡터로 변환되었습니다.")

LLM 테스트 중...
LLM 답변:  안녕하세요! 저는 Google에서 훈련한 대규모 언어 모델입니다.

궁금한 점이 있으시면 언제든지 물어보세요! 😊

임베딩 테스트 중...
임베딩 완료! 텍스트가 3072 차원의 숫자 벡터로 변환되었습니다.


In [8]:
import os
import google.generativeai as genai

print("사용 가능한 모델 목록:")
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

c:\Users\veryf\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\veryf\AppData\Local\Temp\ipykernel_25456\1512797370.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


사용 가능한 모델 목록:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025


### ✅Load & Chunk / Embed & Store

In [9]:
import os
from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    UnstructuredExcelLoader
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# 1. 임베딩 모델 세팅
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# 2. 다중 파일 불러오기 (맞춤형 도어맨 코드)
data_dir = "./data"
docs = []

print("📁 data 폴더의 파일들을 확장자별로 읽어옵니다...\n")

for filename in os.listdir(data_dir):
    file_path = os.path.join(data_dir, filename)
    ext = os.path.splitext(filename)[-1].lower() # 확장자 추출 (.pdf, .txt 등)

    try:
        if ext == ".pdf":
            loader = PyPDFLoader(file_path)
            docs.extend(loader.load())
            print(f"✅ [PDF] {filename} 로드 완료")
            
        elif ext == ".txt":
            # 한글 깨짐 방지를 위해 인코딩을 utf-8로 강제 지정
            loader = TextLoader(file_path, encoding="utf-8") 
            docs.extend(loader.load())
            print(f"✅ [TXT] {filename} 로드 완료")
            
        elif ext == ".xlsx":
            loader = UnstructuredExcelLoader(file_path)
            docs.extend(loader.load())
            print(f"✅ [EXCEL] {filename} 로드 완료")
            
        elif ext == ".hwp":
            print(f"⚠️ [HWP] {filename} -> 파이썬에서 직접 읽기 까다롭습니다. PDF로 변환 후 넣어주세요!")
            
        else:
            print(f"⏩ [SKIP] {filename} -> 지원하지 않는 확장자입니다.")
            
    except Exception as e:
        print(f"❌ {filename} 읽기 실패 (에러: {e})")

print(f"\n📄 총 {len(docs)} 덩어리의 문서 데이터를 성공적으로 불러왔습니다!")

# 3. 문서 조각내기 및 DB 저장 (이전과 동일)
print("\n✂️ 문서를 자르고 Vector DB에 저장합니다...")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits = text_splitter.split_documents(docs)

vectorstore = Chroma.from_documents(
    documents=splits, 
    embedding=embeddings, 
    persist_directory="./chroma_db" 
)
print("✅ 다중 포맷 Vector DB 구축 완료!")

📁 data 폴더의 파일들을 확장자별로 읽어옵니다...

✅ [TXT] ▣ 글솝 졸업요건 기술창업역량(현장실습, 창업교과목, 스타트업창.txt 로드 완료
✅ [TXT] 글로벌소프트웨어융합전공 소속 학생이 이수한 교과목의 교과구분.txt 로드 완료
✅ [PDF] 별첨 도전 K-스타트업 2025 부처 통합 창업경진대회 공고최종.pdf 로드 완료
✅ [PDF] 붙임_교양초과이수자_교과구분변경_관련_매뉴얼-학생용.pdf 로드 완료
✅ [PDF] 졸업 사정 기준2025-글로벌소프트웨어융합전공-2p.pdf 로드 완료
✅ [EXCEL] 참고자료 창업교과목_인정목록_2026.xlsx 로드 완료
✅ [PDF] 참고자료4 창업대체기준창업경진대회 목록 포함2025.pdf 로드 완료
✅ [TXT] 컴퓨터학부(글로벌소프트웨어융합전공).txt 로드 완료
✅ [EXCEL] 학생작성_교양초과이수_교과구분_변경_신청_내역-홍길동-제출일.xlsx 로드 완료
✅ [PDF] 학생작성_교양초과이수_교과구분_변경_신청서-홍길동-제출일자.pdf 로드 완료

📄 총 27 덩어리의 문서 데이터를 성공적으로 불러왔습니다!

✂️ 문서를 자르고 Vector DB에 저장합니다...
✅ 다중 포맷 Vector DB 구축 완료!


### ✅RAG Chain - Flash

In [3]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# 1. 환경변수 및 모델 세팅
load_dotenv(override=True)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0) # 답변을 일관되게 하도록 온도 0 설정
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# 2. 로컬에 저장된 Vector DB 불러오기 (문서를 다시 파싱할 필요가 없습니다!)
print("🧠 로컬에 저장된 Chroma DB를 불러옵니다...")
vectorstore = Chroma(
    persist_directory="./chroma_db", 
    embedding_function=embeddings
)
# 질문과 가장 잘 맞는 문서 조각 4개(k=4)를 찾아오도록 설정
retriever = vectorstore.as_retriever(search_kwargs={"k": 4}) 
print("✅ DB 로드 완료!\n")

# 3. RAG 체인 (프롬프트) 만들기
prompt = ChatPromptTemplate.from_template("""
당신은 경북대학교 컴퓨터학부(글로벌소프트웨어융합전공)의 친절하고 정확한 학사 조교입니다.
아래의 제공된 문서 내용(Context)만 바탕으로 질문에 답변해 주세요.
문서에 없는 내용이라면 지어내지 말고 '제공된 문서에서는 해당 내용을 찾을 수 없습니다'라고 명확히 답해주세요.

Context: {context}

Question: {input}

Answer:
""")
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# 4. 테스트 질문 던지기
question = "2025학년도 졸업 사정 기준을 보면, '다전공'을 하는 학생과 '단일전공'을 하는 학생의 전공 이수 요구 학점이 다를 텐데, 구체적으로 각각 몇 학점씩 들어야 하나요?"

print(f"❓ 질문: {question}")
print("-" * 50)
print("🤖 AI 조교가 문서를 검색하여 답변을 생성하고 있습니다...\n")

response = rag_chain.invoke({"input": question})

print("✨ [최종 답변]")
print(response["answer"])

🧠 로컬에 저장된 Chroma DB를 불러옵니다...
✅ DB 로드 완료!

❓ 질문: 2025학년도 졸업 사정 기준을 보면, '다전공'을 하는 학생과 '단일전공'을 하는 학생의 전공 이수 요구 학점이 다를 텐데, 구체적으로 각각 몇 학점씩 들어야 하나요?
--------------------------------------------------
🤖 AI 조교가 문서를 검색하여 답변을 생성하고 있습니다...

✨ [최종 답변]
안녕하세요, 경북대학교 컴퓨터학부 학사 조교입니다. 2025학년도 졸업 사정 기준에 대해 제공된 문서 내용을 바탕으로 안내해 드리겠습니다.

*   **단일전공을 하는 학생:** 최소 51학점 이상의 전공 학점을 이수해야 합니다.
*   **다전공(복수전공, 연계전공, 융합전공, 부전공)을 하는 학생:** '다전공시 이수학점'을 이수해야 한다고 명시되어 있으나, **제공된 문서에서는 해당 학점의 구체적인 숫자를 찾을 수 없습니다.**
    *   다만, 2023학번부터는 부전공을 1개만 이수하는 경우, 다전공시 전공이수 기준학점에서 소속학과(부) 전공 교과목을 12학점 추가 이수해야 합니다. (이는 2025학년도 졸업생 중 2023학번 이후 학생에게 해당될 수 있습니다.)


### ✅RAG Chain - Pro

In [22]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# 1. 환경변수 및 모델 세팅
load_dotenv(override=True)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-pro", temperature=0) # 답변을 일관되게 하도록 온도 0 설정
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# 2. 로컬에 저장된 Vector DB 불러오기 (문서를 다시 파싱할 필요가 없습니다!)
print("🧠 로컬에 저장된 Chroma DB를 불러옵니다...")
vectorstore = Chroma(
    persist_directory="./chroma_db", 
    embedding_function=embeddings
)
# 질문과 가장 잘 맞는 문서 조각 4개(k=4)를 찾아오도록 설정
retriever = vectorstore.as_retriever(search_kwargs={"k": 4}) 
print("✅ DB 로드 완료!\n")

# 3. RAG 체인 (프롬프트) 만들기
prompt = ChatPromptTemplate.from_template("""
당신은 경북대학교 컴퓨터학부(글로벌소프트웨어융합전공)의 친절하고 정확한 학사 조교입니다.
아래의 제공된 문서 내용(Context)만 바탕으로 질문에 답변해 주세요.
문서에 없는 내용이라면 지어내지 말고 '제공된 문서에서는 해당 내용을 찾을 수 없습니다'라고 명확히 답해주세요.

Context: {context}

Question: {input}

Answer:
""")
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# 4. 테스트 질문 던지기
question = "글로벌소프트웨어융합전공 학생이 졸업하기 위해 이수해야 하는 '총 요구 학점'과 '전공 필수 학점'은 각각 몇 학점인가요?"

print(f"❓ 질문: {question}")
print("-" * 50)
print("🤖 AI 조교가 문서를 검색하여 답변을 생성하고 있습니다...\n")

response = rag_chain.invoke({"input": question})

print("✨ [최종 답변]")
print(response["answer"])

🧠 로컬에 저장된 Chroma DB를 불러옵니다...
✅ DB 로드 완료!

❓ 질문: 글로벌소프트웨어융합전공 학생이 졸업하기 위해 이수해야 하는 '총 요구 학점'과 '전공 필수 학점'은 각각 몇 학점인가요?
--------------------------------------------------
🤖 AI 조교가 문서를 검색하여 답변을 생성하고 있습니다...

✨ [최종 답변]
안녕하세요, 경북대학교 컴퓨터학부(글로벌소프트웨어융합전공) 학사 조교입니다.

제공된 문서에 따르면, 글로벌소프트웨어융합전공 학생의 졸업에 필요한 학점은 다음과 같습니다.

*   **총 요구 학점:** **130학점**
*   **전공 학점:** **51학점** (글솝 교육과정 내 컴퓨터학부 개설 전공)
